# ✍️ Hands-on: prompting & structured output (Domain 4, 20%)

Shape behavior with the system prompt, teach a task with few-shot examples, and get
**machine-parseable** output three ways: XML tags, `output_config.format`, and the
typed `messages.parse()` helper. Live calls on `claude-haiku-4-5`.


In [ ]:
# Setup — run me first
%pip install -q anthropic

import os, getpass, json
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key (input is hidden): ")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"   # cheap + fast for learning; swap to claude-opus-4-8 for the strongest reasoning

def check(name, fn, hint=""):
    try: ok = fn()
    except Exception as e: ok=False; hint=(hint+f"  (raised {type(e).__name__}: {e})").strip()
    print(f"{'\u2705' if ok else '\u274c'}  {name}" + (f"  \u2014 {hint}" if not ok and hint else ""))
    return ok

print("Client ready \u2713   model =", MODEL)


## Exercise 1 — the system prompt shapes behavior
Same question, two personas. The system prompt is your strongest, cheapest steering lever.

**TODO:** set `terse_system` to force a one-word answer.


In [ ]:
def ask(system):
    r = client.messages.create(model=MODEL, max_tokens=200, system=system,
        messages=[{"role": "user", "content": "Is Python dynamically typed?"}])
    return next(b.text for b in r.content if b.type == "text").strip()

# TODO: a system prompt that forces exactly one word (yes/no), no explanation
terse_system = ""   # <-- fill in
print("terse:", repr(ask(terse_system)))
print("default:", ask("You are a helpful assistant.")[:80], "...")


In [ ]:
check("terse system exists", lambda: len(terse_system) > 10)
check("answer is essentially one word",
      lambda: len(ask(terse_system).split()) <= 2, "the system prompt should force brevity")


## Exercise 2 — few-shot examples via message history
Add example user/assistant turns *before* the real question to teach a format. Here:
sentiment as a single symbol.

**TODO:** add a third example so the model learns the exact label set (+ / - / =).


In [ ]:
examples = [
    {"role": "user", "content": "Review: I love it!"},
    {"role": "assistant", "content": "+"},
    {"role": "user", "content": "Review: Broke on day one."},
    {"role": "assistant", "content": "-"},
    # TODO: add a neutral example whose assistant reply is exactly "="
]
msgs = examples + [{"role": "user", "content": "Review: It is fine, nothing special."}]
r = client.messages.create(model=MODEL, max_tokens=10, messages=msgs)
label = next(b.text for b in r.content if b.type == "text").strip()
print("label:", repr(label))


In [ ]:
check("added a neutral '=' example",
      lambda: any(m["role"] == "assistant" and m["content"].strip() == "=" for m in examples))
check("model returned one of + - =", lambda: label in {"+", "-", "="})


## Exercise 3 — typed structured output with messages.parse()
`client.messages.parse(..., output_format=Model)` validates the reply against a Pydantic
model and hands you a typed object — no brittle string parsing, no prefill.

**TODO:** finish the `Contact` model (add an `email: str` field).


In [ ]:
from pydantic import BaseModel

class Contact(BaseModel):
    name: str
    # TODO: add  email: str
    wants_demo: bool

resp = client.messages.parse(model=MODEL, max_tokens=300, output_format=Contact,
    messages=[{"role": "user",
               "content": "Jane Doe (jane@co.com) asked for a demo of the API."}])
contact = resp.parsed_output
print(contact)


In [ ]:
check("Contact has an email field", lambda: "email" in Contact.model_fields)
check("parsed a typed object", lambda: isinstance(contact, Contact) and contact.wants_demo is True)
check("email extracted", lambda: "@" in contact.email)


## Exercise 4 — raw JSON schema with output_config.format
When you don't want Pydantic, constrain the response directly with a JSON schema. The
first content block is then guaranteed-valid JSON.

**TODO:** add the `city` string property to the schema and mark it required.


In [ ]:
schema = {"type": "object",
    "properties": {"summary": {"type": "string"}},   # TODO: add "city": {"type": "string"}
    "required": ["summary"],                          # TODO: add "city"
    "additionalProperties": False}
r = client.messages.create(model=MODEL, max_tokens=300,
    output_config={"format": {"type": "json_schema", "schema": schema}},
    messages=[{"role": "user", "content": "One-line summary of the Eiffel Tower and the city it's in."}])
data = json.loads(next(b.text for b in r.content if b.type == "text"))
print(data)


In [ ]:
check("schema requires city", lambda: "city" in schema["required"] and "city" in schema["properties"])
check("output has both fields", lambda: "summary" in data and "city" in data)


---
**Domain-4 takeaways (exam):** the system prompt is the primary steering tool; few-shot
examples go in the message history; structured output (`messages.parse` / `output_config`)
replaces the now-removed assistant *prefill* for forcing shape.
